# Spring v1 Baseline — Zero-Shot Video Anomaly Detection

| | |
|---|---|
| **Backbone** | CLIP ViT-B/32 |
| **Aggregation** | Max anomaly score across all segments of a video |
| **Threshold** | Grid-searched on validation set (maximise balanced accuracy) |
| **Evaluation** | Precision, Recall, F1, Confusion Matrix on test set |

---

### Expected file structure
```
/data/manifests_new/manifest.csv
/data/segments_new/
    train/  Abuse/Abuse001_x264/frame_0000.jpg ...
    val/    Abuse/Abuse002_x264/frame_0000.jpg ...
    test/   ...
```

### manifest.csv expected columns
`segment_id | video_id | label | split | path`

> Edit **Cell 2 (Configuration)** if your paths or column names differ.

## 0 — Install dependencies
Run once. Skip if already installed.

In [ ]:
# Run once — skip if already installed
import sys
!{sys.executable} -m pip install -q openai-clip torch torchvision scikit-learn matplotlib pandas pillow

## 1 — Configuration
**Edit the paths and column map here if needed.**

In [ ]:
import os, glob as _glob

# ── Auto-detect manifest path ────────────────────────────────────────────────
# Searches common locations. If found, sets MANIFEST_PATH automatically.
# If not found, set MANIFEST_PATH manually below.
_search_roots = ['/', '/home', '/root', '/content', os.getcwd()]
_manifest_found = None
for _root in _search_roots:
    for _dirpath, _dirs, _files in os.walk(_root):
        # Skip large system directories
        _dirs[:] = [d for d in _dirs if d not in
                    {'proc','sys','dev','run','snap','boot','lib','lib64','usr','bin','sbin','etc'}]
        if 'manifest.csv' in _files:
            _manifest_found = os.path.join(_dirpath, 'manifest.csv')
            break
    if _manifest_found:
        break

# ── Paths ────────────────────────────────────────────────────────────────────
# If auto-detect found nothing, set MANIFEST_PATH manually here:
MANIFEST_PATH = _manifest_found if _manifest_found else '/data/manifests_new/manifest.csv'

# SEGMENT_ROOT: folder that contains train/ val/ test/ subdirectories
# Leave as '' to let the path resolver use absolute paths from the manifest
SEGMENT_ROOT  = os.path.dirname(os.path.dirname(MANIFEST_PATH)).replace('manifests_new','segments_new') if _manifest_found else '/data/segments_new'

RESULTS_DIR   = 'spring_v1_results'

print(f'MANIFEST_PATH : {MANIFEST_PATH}')
print(f'  exists       : {os.path.exists(MANIFEST_PATH)}')
print(f'SEGMENT_ROOT  : {SEGMENT_ROOT}')
print(f'  exists       : {os.path.exists(SEGMENT_ROOT)}')

# ── Column map ───────────────────────────────────────────────────────────────
# Keys = what the code expects | Values = actual column names in your CSV
# Your manifest columns (update right side if your CSV uses different names):
COLUMN_MAP = {
    'video_id' : 'video_id',   # unique video identifier
    'label'    : 'class',      # ← 'class' in your manifest = category name
    'split'    : 'split',      # train/val/test
    'path'     : 'path',       # segment folder path
}

# ── Prompts  (v1 — intentionally simple) ─────────────────────────────────────
ANOMALY_PROMPTS = [
    'a robbery happening',
    'a person fighting',
    'a violent action',
    'people fighting or attacking each other',
    'criminal activity in a surveillance video',
]

NORMAL_PROMPTS = [
    'people walking normally',
    'a peaceful street scene',
    'normal activity in a public place',
    'ordinary daily life',
]

print('\nConfiguration loaded.')

## 2 — Imports & Device

In [ ]:
import os, glob, warnings
import numpy as np
import pandas as pd
from PIL import Image
import torch
import clip
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, balanced_accuracy_score
)
import matplotlib.pyplot as plt
%matplotlib inline
warnings.filterwarnings("ignore")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"Device : {DEVICE}")
print(f"Results: ./{RESULTS_DIR}/")

## 3 — Load & Inspect Manifest

In [ ]:
# Try reading with utf-8, fallback to latin-1
try:
    raw = pd.read_csv(MANIFEST_PATH, encoding='utf-8')
except UnicodeDecodeError:
    raw = pd.read_csv(MANIFEST_PATH, encoding='latin-1')

print(f'Columns : {raw.columns.tolist()}')
print(f'Shape   : {raw.shape}')
raw.head(3)

In [ ]:
# ── DIAGNOSTIC: inspect manifest structure ────────────────────────────────
print('=== RAW COLUMN NAMES ===')
print(raw.columns.tolist())
print()
print('=== DUPLICATE COLUMNS? ===')
dups = raw.columns[raw.columns.duplicated()].tolist()
print('Duplicates:', dups if dups else 'None')
print()
print('=== DTYPES ===')
print(raw.dtypes)
print()
print('=== FIRST 5 ROWS ===')
display(raw.head())
print()
print('=== UNIQUE VALUES IN EACH COLUMN (first 10) ===')
for col in raw.columns:
    try:
        vals = raw[col].dropna().unique()[:10].tolist()
        print(f'  {col!r:25s}: {vals}')
    except Exception as e:
        print(f'  {col!r:25s}: ERROR - {e}')


In [ ]:
# ── Fix duplicate columns if any ────────────────────────────────────────────
if raw.columns.duplicated().any():
    print('Duplicate columns found — keeping first occurrence')
    raw = raw.loc[:, ~raw.columns.duplicated()]

df = raw.copy()

# ── Map columns to standard names ────────────────────────────────────────────
# Your manifest has:
#   'class'  → human-readable category name  (Abuse, Stealing, NormalVideos …)
#   'label'  → binary integer  (1=anomaly, 0=normal)
#   'split'  → val / test
#   'path'   → absolute segment path

# Rename 'class' → 'label_name' so we keep both
if 'class' in df.columns:
    df = df.rename(columns={'class': 'label_name'})

# Rename 'label' (0/1) → 'is_anomaly' to avoid confusion
if 'label' in df.columns:
    df = df.rename(columns={'label': 'is_anomaly_raw'})

# Create the 'label' column the rest of the code expects (category name string)
if 'label_name' in df.columns:
    df['label'] = df['label_name'].astype(str).str.strip()
elif 'is_anomaly_raw' in df.columns:
    # Fallback: derive name from binary
    df['label'] = df['is_anomaly_raw'].apply(lambda x: 'Anomaly' if int(x)==1 else 'Normal')

# Normalise split: 'val' → 'validation'
df['split'] = df['split'].astype(str).str.strip().str.lower()
df['split'] = df['split'].replace({'val': 'validation'})

# Validate required columns
required = {'video_id', 'label', 'split', 'path'}
missing  = required - set(df.columns)
if missing:
    print('MISSING:', missing)
    print('Available:', df.columns.tolist())
    raise ValueError(f'Missing columns: {missing}')

print('Columns now:', df.columns.tolist())
print()
print('Segments per split:')
print(df['split'].value_counts().to_string())
print()
print('Segments per label:')
print(df['label'].value_counts().to_string())
print(f'\nTotal segments: {len(df)}')

## 4 — Load CLIP ViT-B/32 & Encode Prompts

In [ ]:
print("Loading CLIP ViT-B/32 …")
model, preprocess = clip.load("ViT-B/32", device=DEVICE)
model.eval()
print("Done.")

In [ ]:
def encode_prompts(model, prompts):
    """Encode a list of text prompts → mean-pooled unit-norm embedding."""
    tokens = clip.tokenize(prompts).to(DEVICE)
    with torch.no_grad():
        embs = model.encode_text(tokens).float()
    embs = embs / embs.norm(dim=-1, keepdim=True)
    mean = embs.mean(0)
    return mean / mean.norm()

anom_emb = encode_prompts(model, ANOMALY_PROMPTS)
norm_emb = encode_prompts(model, NORMAL_PROMPTS)

print(f"Anomaly embedding shape : {anom_emb.shape}")
print(f"Normal  embedding shape : {norm_emb.shape}")
print(f"Anomaly prompts used    : {len(ANOMALY_PROMPTS)}")
print(f"Normal  prompts used    : {len(NORMAL_PROMPTS)}")

## 5 — Segment-Level Inference

For each segment:
1. Load up to 16 frames from the segment folder
2. Encode every frame with CLIP image encoder
3. Mean-pool frame embeddings → segment embedding
4. Cosine similarity vs. anomaly & normal text embeddings

In [ ]:
def resolve_path(raw_path):
    """Resolve segment path: absolute → as-is; relative → prepend SEGMENT_ROOT;
    name only → walk SEGMENT_ROOT to find it."""
    if os.path.isabs(raw_path) and os.path.exists(raw_path):
        return raw_path
    joined = os.path.join(SEGMENT_ROOT, raw_path)
    if os.path.exists(joined):
        return joined
    # Last resort: search by folder name
    basename = os.path.basename(raw_path.rstrip("/\\"))
    for root, dirs, _ in os.walk(SEGMENT_ROOT):
        if basename in dirs:
            return os.path.join(root, basename)
    return joined  # best guess


def load_frames(seg_path, max_frames=16):
    """Load up to max_frames images from a segment folder.
    Returns list of preprocessed tensors or None."""
    full = resolve_path(seg_path)
    if os.path.isdir(full):
        frames = []
        for ext in ("*.jpg", "*.jpeg", "*.png"):
            frames.extend(sorted(glob.glob(os.path.join(full, ext))))
        frames = frames[:max_frames]
        if not frames:
            return None
        return [preprocess(Image.open(f).convert("RGB")) for f in frames]
    if os.path.isfile(full):
        try:
            return [preprocess(Image.open(full).convert("RGB"))]
        except Exception:
            return None
    return None


def segment_score(frames):
    """Encode frames, mean-pool, return (score_anomaly, score_normal)."""
    batch = torch.stack(frames).to(DEVICE)
    with torch.no_grad():
        img_embs = model.encode_image(batch).float()
    img_embs = img_embs / img_embs.norm(dim=-1, keepdim=True)
    seg_emb  = img_embs.mean(0)
    seg_emb  = seg_emb / seg_emb.norm()
    return (seg_emb @ anom_emb).item(), (seg_emb @ norm_emb).item()


print("Helper functions defined.")

In [ ]:
records = []
n = len(df)

for i, (_, row) in enumerate(df.iterrows()):
    frames = load_frames(row["path"])
    if frames is None:
        print(f"  [WARN] Cannot load: {row['path']}")
        sa, sn = np.nan, np.nan
    else:
        sa, sn = segment_score(frames)

    records.append({
        "video_id"     : row["video_id"],
        "label"        : row["label"],
        "split"        : row["split"],
        "score_anomaly": sa,
        "score_normal" : sn,
        "score_diff"   : (sa - sn) if not np.isnan(sa) else np.nan,
    })

    if (i + 1) % 50 == 0 or (i + 1) == n:
        print(f"  [{i+1}/{n}] processed")

seg_df = pd.DataFrame(records)
seg_csv = os.path.join(RESULTS_DIR, "segment_scores.csv")
seg_df.to_csv(seg_csv, index=False)
print(f"\nSegment scores saved → {seg_csv}")
seg_df.head()

## 6 — Video-Level Aggregation

**Spring v1 strategy:** `video_score = max(score_anomaly)` across all segments of a video.

In [ ]:
seg_df['video_id'] = seg_df['video_id'].astype(str).str.strip()
seg_df['split']    = seg_df['split'].astype(str).str.strip()
seg_df['label']    = seg_df['label'].astype(str).str.strip()

# Bring is_anomaly_raw into seg_df if available (more reliable than text label)
if 'is_anomaly_raw' in df.columns:
    id_map = df.set_index('video_id')['is_anomaly_raw'].to_dict()
else:
    id_map = None

rows = []
for (vid, split), grp in seg_df.groupby(['video_id', 'split']):
    grp = grp.dropna(subset=['score_anomaly'])
    if grp.empty:
        continue
    label = grp['label'].iloc[0]
    # Prefer binary label from manifest over text parsing
    if id_map is not None and vid in id_map:
        is_anom = int(id_map[vid])
    else:
        is_anom = int(label.lower() not in ('normal', 'normalvideos', 'normal videos'))
    rows.append({
        'video_id'   : vid,
        'split'      : split,
        'label'      : label,
        'is_anomaly' : is_anom,
        'video_score': grp['score_anomaly'].max(),
        'n_segments' : len(grp),
    })

video_df = pd.DataFrame(rows)
video_df.to_csv(os.path.join(RESULTS_DIR, 'video_scores.csv'), index=False)

print('Videos per split and class:')
print(video_df.groupby(['split', 'is_anomaly']).size().to_string())
video_df.head()

## 7 — Threshold Optimisation (Validation Set)

Grid-search over 300 candidate thresholds. Metric: **Balanced Accuracy** (equally weights recall and specificity).

In [ ]:
val = video_df[video_df["split"] == "validation"]
if val.empty:
    raise ValueError("No validation rows found. Check manifest 'split' values.")

y_val  = val["is_anomaly"].values
sc_val = val["video_score"].values

best_t, best_ba = 0.0, -1.0
thresholds, bal_accs = [], []

for t in np.linspace(sc_val.min(), sc_val.max(), 300):
    ba = balanced_accuracy_score(y_val, (sc_val >= t).astype(int))
    thresholds.append(t)
    bal_accs.append(ba)
    if ba > best_ba:
        best_ba, best_t = ba, t

print(f"Best threshold  τ  = {best_t:.4f}")
print(f"Val Balanced Acc   = {best_ba:.4f}")

# Plot threshold sweep
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(thresholds, bal_accs, color="steelblue", linewidth=1.5)
ax.axvline(best_t, color="red", linestyle="--", label=f"τ = {best_t:.4f}")
ax.set(xlabel="Threshold", ylabel="Balanced Accuracy",
       title="Threshold Sweep — Validation Set (Spring v1)")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "threshold_sweep.png"), dpi=150)
plt.show()

## 8 — Evaluation

In [ ]:
def evaluate_split(video_df, threshold, split):
    sub = video_df[video_df["split"] == split]
    if sub.empty:
        print(f"No rows for split='{split}'")
        return {}
    y    = sub["is_anomaly"].values
    pred = (sub["video_score"].values >= threshold).astype(int)

    prec = precision_score(y, pred, zero_division=0)
    rec  = recall_score(y, pred, zero_division=0)
    f1   = f1_score(y, pred, zero_division=0)
    bac  = balanced_accuracy_score(y, pred)
    cm   = confusion_matrix(y, pred)
    try:
        auc = roc_auc_score(y, sub["video_score"].values)
    except Exception:
        auc = float("nan")

    tn, fp, fn, tp = cm.ravel() if cm.shape == (2, 2) else (0, 0, 0, 0)
    return dict(split=split, threshold=threshold,
                precision=prec, recall=rec, f1=f1,
                balanced_accuracy=bac, roc_auc=auc,
                TP=int(tp), FP=int(fp), FN=int(fn), TN=int(tn))

print("evaluate_split() defined.")

In [ ]:
# ── Validation set ────────────────────────────────────────────────────────────
val_metrics = evaluate_split(video_df, best_t, "validation")

print("VALIDATION SET")
print(f"  Threshold         : {val_metrics['threshold']:.4f}")
print(f"  Precision         : {val_metrics['precision']:.4f}")
print(f"  Recall            : {val_metrics['recall']:.4f}")
print(f"  F1-Score          : {val_metrics['f1']:.4f}")
print(f"  Balanced Accuracy : {val_metrics['balanced_accuracy']:.4f}")
print(f"  ROC-AUC           : {val_metrics['roc_auc']:.4f}")
print(f"  TP={val_metrics['TP']}  FP={val_metrics['FP']}  FN={val_metrics['FN']}  TN={val_metrics['TN']}")

In [ ]:
# ── Test set  (threshold fixed from validation — no re-tuning) ────────────────
test_metrics = evaluate_split(video_df, best_t, "test")

print("TEST SET")
print(f"  Threshold         : {test_metrics['threshold']:.4f}")
print(f"  Precision         : {test_metrics['precision']:.4f}")
print(f"  Recall            : {test_metrics['recall']:.4f}")
print(f"  F1-Score          : {test_metrics['f1']:.4f}")
print(f"  Balanced Accuracy : {test_metrics['balanced_accuracy']:.4f}")
print(f"  ROC-AUC           : {test_metrics['roc_auc']:.4f}")
print(f"  TP={test_metrics['TP']}  FP={test_metrics['FP']}  FN={test_metrics['FN']}  TN={test_metrics['TN']}")

## 9 — Plots

In [ ]:
def plot_confusion_matrix(m, split):
    arr = np.array([[m["TN"], m["FP"]], [m["FN"], m["TP"]]])
    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(arr, cmap="Blues")
    plt.colorbar(im, ax=ax)
    ax.set(xticks=[0,1], yticks=[0,1],
           xticklabels=["Normal","Anomaly"],
           yticklabels=["Normal","Anomaly"],
           xlabel="Predicted", ylabel="Actual",
           title=f"Confusion Matrix — {split.capitalize()}\n"
                 f"(Spring v1, ViT-B/32, Max Score)")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(arr[i,j]), ha="center", va="center",
                    fontsize=14, fontweight="bold",
                    color="white" if arr[i,j] > arr.max()/2 else "black")
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, f"confusion_matrix_{split}.png"), dpi=150)
    plt.show()

plot_confusion_matrix(val_metrics, "validation")
plot_confusion_matrix(test_metrics, "test")

In [ ]:
def plot_score_distribution(video_df, threshold, split):
    sub  = video_df[video_df["split"] == split]
    norm = sub[sub["is_anomaly"]==0]["video_score"]
    anom = sub[sub["is_anomaly"]==1]["video_score"]
    bins = np.linspace(sub["video_score"].min()-0.005,
                       sub["video_score"].max()+0.005, 30)
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(norm, bins=bins, alpha=0.6, label="Normal",  color="steelblue")
    ax.hist(anom, bins=bins, alpha=0.6, label="Anomaly", color="darkorange")
    ax.axvline(threshold, color="red", linestyle="--",
               linewidth=1.5, label=f"τ = {threshold:.4f}")
    ax.set(xlabel="Max Anomaly Score", ylabel="Count",
           title=f"Score Distribution — {split.capitalize()} (Spring v1, ViT-B/32)")
    ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, f"score_distribution_{split}.png"), dpi=150)
    plt.show()

plot_score_distribution(video_df, best_t, "validation")
plot_score_distribution(video_df, best_t, "test")

In [ ]:
results = [val_metrics, test_metrics]
res_df  = pd.DataFrame(results)[["split","precision","recall","f1"]]
x = np.arange(len(res_df)); w = 0.25

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(x-w, res_df["precision"], w, label="Precision", color="steelblue")
ax.bar(x,   res_df["recall"],   w, label="Recall",    color="darkorange")
ax.bar(x+w, res_df["f1"],       w, label="F1",        color="seagreen")
ax.set_xticks(x)
ax.set_xticklabels([s.capitalize() for s in res_df["split"]])
ax.set_ylim(0, 1.15); ax.set_ylabel("Score")
ax.set_title("Spring v1 Baseline — Performance Summary (ViT-B/32, Max Score)")
ax.legend()
for bar in ax.patches:
    h = bar.get_height()
    if h > 0.01:
        ax.text(bar.get_x()+bar.get_width()/2, h+0.01,
                f"{h:.2f}", ha="center", va="bottom", fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "performance_summary.png"), dpi=150)
plt.show()

## 10 — Save Results & Print Table Row

In [ ]:
metrics_csv = os.path.join(RESULTS_DIR, "metrics_summary.csv")
pd.DataFrame([val_metrics, test_metrics]).to_csv(metrics_csv, index=False)
print(f"Metrics saved → {metrics_csv}")

tm = test_metrics
print("\n" + "═"*54)
print("  SPRING v1 — FINAL TEST RESULTS")
print("═"*54)
print(f"  Backbone    : ViT-B/32")
print(f"  Aggregation : Max segment anomaly score")
print(f"  Threshold   : {best_t:.4f}")
print(f"  Precision   : {tm['precision']:.4f}")
print(f"  Recall      : {tm['recall']:.4f}")
print(f"  F1-Score    : {tm['f1']:.4f}")
print(f"  ROC-AUC     : {tm['roc_auc']:.4f}")
print(f"  TP={tm['TP']}  FP={tm['FP']}  FN={tm['FN']}  TN={tm['TN']}")
print("═"*54)
print(f"\n  → Table 10 row:")
print(f"  Spring v1 | ViT-B/32, max score | "
      f"{tm['precision']:.2f} | {tm['recall']:.2f} | {tm['f1']:.2f}")